# Dr. Splat: Directly Referring 3D Gaussian Splatting
**CVPR 2025 Highlight**

1. 환경 설정
2. 데이터셋 다운로드 (LeRF-OVS)
3. 기본 3DGS 학습 (30K iterations)
4. Feature 추출 (SAM + OpenCLIP)
5. Dr. Splat Feature Registration
6. 시각화 (PCA / Activation)

---
## 0. Configuration

In [ ]:
import os, shutil, glob

WORK_DIR = "/content/drsplat"
MY_GITHUB_REPO = "https://github.com/BAEJUNHAK/Dr-Splat.git"

SCENE_NAME = "ramen"  # figurines / ramen / teatime / waldo_kitchen

DATASET_ROOT = f"{WORK_DIR}/data/lerf_ovs"
SCENE_PATH = f"{DATASET_ROOT}/{SCENE_NAME}"

DRSPLAT_REPO = f"{WORK_DIR}/Dr-Splat"
GS_REPO = f"{WORK_DIR}/gaussian-splatting"

GS_OUTPUT_PATH = f"{WORK_DIR}/output/3dgs_{SCENE_NAME}"
GS_CHECKPOINT = f"{GS_OUTPUT_PATH}/chkpnt30000.pth"
DRSPLAT_OUTPUT_PATH = f"{WORK_DIR}/output/drsplat_{SCENE_NAME}"

# Google Drive 백업 경로
DRIVE_SAVE_DIR = "/content/drive/MyDrive/drsplat_results"

GPU_ID = 0
TOPK = 40

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Scene: {SCENE_NAME}")
print(f"Scene path: {SCENE_PATH}")

---
## 1. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 2. 환경 설정

### 2-1. 레포 클론

In [ ]:
# Dr-Splat (본인 GitHub) - .git 없으면 삭제 후 재클론
if os.path.exists(DRSPLAT_REPO) and os.path.isdir(os.path.join(DRSPLAT_REPO, '.git')):
    print("Dr-Splat exists. Pulling latest...")
    !cd {DRSPLAT_REPO} && git pull origin main
else:
    !rm -rf {DRSPLAT_REPO}
    !cd {WORK_DIR} && git clone {MY_GITHUB_REPO} --recursive
    print("Dr-Splat cloned!")

# 서브모듈 초기화 (중요!)
!cd {DRSPLAT_REPO} && git submodule update --init --recursive
print("\n=== submodules ===")
!ls {DRSPLAT_REPO}/submodules/

In [ ]:
# 원본 3DGS (기본 학습용)
if not os.path.exists(GS_REPO):
    !cd {WORK_DIR} && git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive
    print("3DGS cloned!")
else:
    print("3DGS already exists.")

### 2-2. 패키지 설치
Colab 기본 PyTorch (CUDA 12.x)를 그대로 사용합니다.

In [ ]:
!pip install plyfile tqdm scipy scikit-image matplotlib opencv-python-headless -q
!pip install open-clip-torch faiss-cpu kmeans_pytorch -q
!pip install tensorboard tyro jaxtyping gdown -q

### 2-3. CUDA 서브모듈 빌드
> 빌드에 몇 분 걸릴 수 있습니다.

In [ ]:
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["CUDA_PATH"] = "/usr/local/cuda"

# 원본 3DGS 서브모듈 (기본 학습용)
!pip install {GS_REPO}/submodules/diff-gaussian-rasterization
!pip install {GS_REPO}/submodules/simple-knn

---
## 3. Google Drive 마운트 & 데이터셋 다운로드
체크포인트 자동 저장/복원을 위해 먼저 Drive를 마운트합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Drive backup dir: {DRIVE_SAVE_DIR}")

### 3-1. LeRF-OVS 다운로드

| 소스 | 링크 |
|------|------|
| LangSplat Google Drive | [링크](https://drive.google.com/file/d/1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt/view) |
| OpenGaussian Baidu | [링크](https://pan.baidu.com/s/1B_tGYla5dWyJRu3jTNTMvA?pwd=u5iy) |

방법 1부터 시도하세요.

#### 방법 1: gdown (권장)

In [ ]:
import gdown, shutil

DOWNLOAD_DIR = f"{WORK_DIR}/data"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

GDRIVE_ID = "1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt"
ZIP_PATH = f"{DOWNLOAD_DIR}/lerf_ovs.zip"

if not os.path.exists(SCENE_PATH):
    print("Downloading LeRF-OVS ...")
    try:
        gdown.download(id=GDRIVE_ID, output=ZIP_PATH, quiet=False, fuzzy=True)
        if os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 1_000_000:
            print(f"Download: {os.path.getsize(ZIP_PATH) / 1e9:.2f} GB")
            !cd {DOWNLOAD_DIR} && unzip -q -o {ZIP_PATH}
            !rm -f {ZIP_PATH}

            # 중첩 폴더 자동 처리: data/lerf_ovs/lerf_ovs/ -> data/lerf_ovs/
            nested = os.path.join(DATASET_ROOT, "lerf_ovs")
            if os.path.isdir(nested) and not os.path.isdir(os.path.join(DATASET_ROOT, SCENE_NAME)):
                tmp = f"{DOWNLOAD_DIR}/_lerf_tmp"
                shutil.move(nested, tmp)
                shutil.rmtree(DATASET_ROOT)
                shutil.move(tmp, DATASET_ROOT)
                print("Nested folder fixed!")
            print("Done!")
        else:
            print("[WARN] gdown 실패. 방법 2를 사용하세요.")
            !rm -f {ZIP_PATH}
    except Exception as e:
        print(f"[ERROR] {e}\n방법 2를 사용하세요.")
else:
    print(f"Dataset exists: {SCENE_PATH}")

#### 방법 2: Google Drive 마운트 (gdown 실패시)

In [ ]:
# === 주석 해제 후 실행 ===

# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ZIP = "/content/drive/MyDrive/lerf_ovs.zip"  # <-- 경로 수정
# !cp {DRIVE_ZIP} {DOWNLOAD_DIR}/
# !cd {DOWNLOAD_DIR} && unzip -q -o lerf_ovs.zip && rm lerf_ovs.zip

### 3-2. 데이터 구조 검증

In [ ]:
print("=== Available scenes ===")
if os.path.exists(DATASET_ROOT):
    for d in sorted(os.listdir(DATASET_ROOT)):
        full = os.path.join(DATASET_ROOT, d)
        if os.path.isdir(full):
            has_img = os.path.isdir(os.path.join(full, 'images'))
            has_sp = os.path.isdir(os.path.join(full, 'sparse', '0'))
            n = len(os.listdir(os.path.join(full, 'images'))) if has_img else 0
            print(f"  {d:20s} images:{n:3d}  sparse:{'OK' if has_sp else 'X'}")
else:
    print(f"[ERROR] {DATASET_ROOT} not found!")

# GT label 확인 (평가에 필수)
LABEL_DIR = os.path.join(DATASET_ROOT, "label")
if os.path.isdir(LABEL_DIR):
    label_scenes = [d for d in os.listdir(LABEL_DIR) if os.path.isdir(os.path.join(LABEL_DIR, d))]
    print(f"\n=== GT Labels ===")
    print(f"  label/ found: {sorted(label_scenes)}")
else:
    print(f"\n[WARN] label/ not found!")
    # Ref-lerf.zip이 있으면 자동 압축 해제
    ref_zip = os.path.join(DATASET_ROOT, "Ref-lerf.zip")
    if os.path.exists(ref_zip):
        print(f"  Found {ref_zip}, extracting...")
        !cd {DATASET_ROOT} && unzip -q -o {ref_zip}
        print("  Done! Checking again...")
        if os.path.isdir(LABEL_DIR):
            print(f"  label/ now available!")
        else:
            # Ref-lerf/ 안에 label이 있을 수 있음
            !ls {DATASET_ROOT}/Ref-lerf*/ 2>/dev/null
            print("  [WARN] label/ still not found. 수동 확인 필요.")
    else:
        print("  Ref-lerf.zip도 없습니다.")
        print("  평가(Section 8)를 하려면 label/ 폴더가 필요합니다.")

def verify(path, name):
    checks = {
        'images/': os.path.isdir(os.path.join(path, 'images')),
        'sparse/0/cameras.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'cameras.bin')),
        'sparse/0/images.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'images.bin')),
        'sparse/0/points3D.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'points3D.bin')),
    }
    ok = all(checks.values())
    print(f"\n{'PASS' if ok else 'FAIL'} - {name}")
    for k, v in checks.items():
        print(f"  {'[O]' if v else '[X]'} {k}")

verify(SCENE_PATH, SCENE_NAME)

### 3-3. SAM 체크포인트 다운로드

In [ ]:
SAM_CKPT = f"{DRSPLAT_REPO}/ckpts/sam_vit_h_4b8939.pth"
os.makedirs(f"{DRSPLAT_REPO}/ckpts", exist_ok=True)

if not os.path.exists(SAM_CKPT):
    print("Downloading SAM ViT-H...")
    !wget -q -P {DRSPLAT_REPO}/ckpts/ https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print("Done!")
else:
    print("SAM checkpoint exists.")
!ls -lh {SAM_CKPT}

---
## 4. 기본 3DGS 학습 (30K iterations)
> T4 기준 약 20~40분 소요

In [ ]:
os.makedirs(GS_OUTPUT_PATH, exist_ok=True)

# Drive에서 체크포인트 확인
drive_gs_dir = os.path.join(DRIVE_SAVE_DIR, f"3dgs_{SCENE_NAME}")
drive_ckpt = os.path.join(drive_gs_dir, "chkpnt30000.pth")

if os.path.exists(GS_CHECKPOINT):
    # 로컬에 이미 있음
    print(f"Checkpoint exists (local): {GS_CHECKPOINT}")

elif os.path.exists(drive_ckpt):
    # Drive에서 복원
    print(f"Restoring checkpoint from Drive: {drive_gs_dir}")
    shutil.copytree(drive_gs_dir, GS_OUTPUT_PATH, dirs_exist_ok=True)
    print(f"Restored! {GS_CHECKPOINT}")

else:
    # 학습 실행
    print("Starting 3DGS training (30K iterations)...")
    !cd {GS_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python train.py \
        -s {SCENE_PATH} \
        -m {GS_OUTPUT_PATH} \
        --iterations 30000 \
        --save_iterations 30000 \
        --checkpoint_iterations 30000 \
        --eval
    print("Training done!")

    # 학습 완료 후 Drive에 자동 저장
    if os.path.exists(GS_CHECKPOINT):
        print(f"\nSaving checkpoint to Drive: {drive_gs_dir}")
        shutil.copytree(GS_OUTPUT_PATH, drive_gs_dir, dirs_exist_ok=True)
        print("Saved!")

In [ ]:
!ls -lh {GS_OUTPUT_PATH}/chkpnt*.pth 2>/dev/null || echo "No checkpoint found!"

---
## 5. Feature 추출 (SAM + OpenCLIP)
> **중요**: langsplat-rasterization으로 교체합니다.

In [ ]:
# 서브모듈 확인
!cd {DRSPLAT_REPO} && git submodule update --init --recursive
!ls {DRSPLAT_REPO}/submodules/

# 원본 rasterizer 제거 -> langsplat 버전 설치
!pip uninstall diff-gaussian-rasterization -y -q 2>/dev/null
!pip install {DRSPLAT_REPO}/submodules/langsplat-rasterization
!pip install {DRSPLAT_REPO}/submodules/segment-anything-langsplat

In [ ]:
# CLIP 레포
CLIP_DIR = f"{DRSPLAT_REPO}/CLIP"
if not os.path.exists(CLIP_DIR):
    !cd {DRSPLAT_REPO} && git clone https://github.com/openai/CLIP.git
else:
    print("CLIP exists.")

In [ ]:
LF_DIR = f"{SCENE_PATH}/language_features"
drive_lf_dir = os.path.join(DRIVE_SAVE_DIR, f"language_features_{SCENE_NAME}")

if os.path.exists(LF_DIR) and len(os.listdir(LF_DIR)) > 0:
    # 로컬에 이미 있음
    print(f"Language features exist (local): {LF_DIR}")
    !ls {LF_DIR} | head -10

elif os.path.exists(drive_lf_dir) and len(os.listdir(drive_lf_dir)) > 0:
    # Drive에서 복원
    print(f"Restoring language_features from Drive: {drive_lf_dir}")
    shutil.copytree(drive_lf_dir, LF_DIR, dirs_exist_ok=True)
    print(f"Restored! ({len(os.listdir(LF_DIR))} files)")

else:
    # 전처리 실행
    os.makedirs(LF_DIR, exist_ok=True)
    print("Starting feature extraction (SAM + OpenCLIP)...")
    print("This takes several hours on T4.")
    !cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python preprocessing.py \
        --dataset_path {SCENE_PATH} \
        --sam_ckpt_path {SAM_CKPT}
    print("Done!")

In [ ]:
import numpy as np
lf_files = sorted(os.listdir(LF_DIR))
f_files = [f for f in lf_files if f.endswith('_f.npy')]
print(f"Total: {len(lf_files)} files, {len(f_files)} features")
if f_files:
    print(f"Shape: {np.load(os.path.join(LF_DIR, f_files[0])).shape}")

In [ ]:
# language_features를 Drive에 백업 (전처리 5시간 날리지 않기 위해)
drive_lf_dir = os.path.join(DRIVE_SAVE_DIR, f"language_features_{SCENE_NAME}")
if os.path.exists(LF_DIR) and len(os.listdir(LF_DIR)) > 0:
    print(f"Saving language_features to Drive: {drive_lf_dir}")
    shutil.copytree(LF_DIR, drive_lf_dir, dirs_exist_ok=True)
    print(f"Saved! ({len(os.listdir(drive_lf_dir))} files)")
else:
    print("[WARN] language_features not found, nothing to save")

---
## 6. Dr. Splat Feature Registration
Majority Voting + PQ 압축 (gradient 최적화 아님)

In [ ]:
PQ_INDEX = f"{DRSPLAT_REPO}/ckpts/pq_index.faiss"
if os.path.exists(PQ_INDEX):
    print(f"PQ index: {PQ_INDEX}")
else:
    print("[ERROR] PQ index not found!")
    !ls -la {DRSPLAT_REPO}/ckpts/

In [ ]:
from argparse import Namespace
import faiss

# Dr. Splat 출력 경로 계산 (train.py가 suffix를 붙임)
_pq_index = faiss.read_index(f"{DRSPLAT_REPO}/ckpts/pq_index.faiss")
try:
    _suffix = f"_{1}_{('pq_openclip')}_topk{TOPK}_weight_{_pq_index.coarsecode_size()+_pq_index.code_size}"
except:
    _suffix = f"_{1}_{('pq_openclip')}_topk{TOPK}_weight_{_pq_index.code_size}"
DRSPLAT_ACTUAL_PATH = DRSPLAT_OUTPUT_PATH + _suffix
drive_drsplat_dir = os.path.join(DRIVE_SAVE_DIR, os.path.basename(DRSPLAT_ACTUAL_PATH))

# cfg_args 생성 함수
def ensure_cfg_args(model_path):
    cfg_path = os.path.join(model_path, "cfg_args")
    if not os.path.exists(cfg_path):
        os.makedirs(model_path, exist_ok=True)
        cfg = Namespace(
            sh_degree=3, source_path=SCENE_PATH, model_path=model_path,
            language_features_name='language_features_dim3', images='images',
            resolution=-1, white_background=False, feature_level=1,
            data_device='cuda', eval=True
        )
        with open(cfg_path, 'w') as f:
            f.write(str(cfg))
        print(f"cfg_args created: {cfg_path}")

# 로컬에 결과 있는지 확인
local_ckpt = os.path.join(DRSPLAT_ACTUAL_PATH, "chkpnt0.pth")
drive_ckpt = os.path.join(drive_drsplat_dir, "chkpnt0.pth")

if os.path.exists(local_ckpt):
    print(f"Dr. Splat result exists (local): {DRSPLAT_ACTUAL_PATH}")
    ensure_cfg_args(DRSPLAT_ACTUAL_PATH)

elif os.path.exists(drive_ckpt):
    print(f"Restoring Dr. Splat from Drive: {drive_drsplat_dir}")
    shutil.copytree(drive_drsplat_dir, DRSPLAT_ACTUAL_PATH, dirs_exist_ok=True)
    ensure_cfg_args(DRSPLAT_ACTUAL_PATH)
    print("Restored!")

else:
    print("Starting Dr. Splat feature registration...")
    !cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python train.py \
        -s {SCENE_PATH} \
        -m {DRSPLAT_OUTPUT_PATH} \
        --start_checkpoint {GS_CHECKPOINT} \
        --feature_level 1 \
        --name_extra pq_openclip \
        --use_pq \
        --pq_index {PQ_INDEX} \
        --port 55560 \
        --topk {TOPK} \
        --eval
    print("Done!")
    ensure_cfg_args(DRSPLAT_ACTUAL_PATH)

    # Drive에 자동 저장
    if os.path.exists(DRSPLAT_ACTUAL_PATH):
        print(f"Saving to Drive: {drive_drsplat_dir}")
        shutil.copytree(DRSPLAT_ACTUAL_PATH, drive_drsplat_dir, dirs_exist_ok=True)
        print("Saved!")

In [ ]:
DRSPLAT_TRAINED_PATH = DRSPLAT_ACTUAL_PATH
print(f"Dr. Splat output: {DRSPLAT_TRAINED_PATH}")
!ls -la {DRSPLAT_TRAINED_PATH}/ 2>/dev/null || print("[WARN] Directory not found")

---
## 7. 시각화

### 7-1. PCA Feature Visualization

In [ ]:
!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python render_pca.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_TRAINED_PATH} \
    --pq_index {PQ_INDEX} \
    --feature_level 1 \
    -l language_features_dim3

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

pca_candidates = glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/renders_pca", recursive=True)
if pca_candidates:
    pca_images = sorted(glob.glob(f"{pca_candidates[0]}/*.png"))[:8]
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, p in zip(axes.flat, pca_images):
        ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8); ax.axis('off')
    plt.suptitle('PCA Feature Visualization', fontsize=16); plt.tight_layout(); plt.show()
else:
    print("PCA output not found")

### 7-2. Activation (텍스트 쿼리)

In [ ]:
TEXT_QUERY = "ramen"
SAVE_LABEL = "ramen_test"
THRESHOLD = 0.5

!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python render_activation.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_TRAINED_PATH} \
    --semantic_model sam \
    --feature_level 1 \
    --pq_index {PQ_INDEX} \
    --img_label "{TEXT_QUERY}" \
    --img_save_label "{SAVE_LABEL}" \
    --threshold {THRESHOLD} \
    -l language_features_dim3

In [ ]:
# 결과 시각화: colormap / binary / side-by-side
for viz_type in ["colormap", "binary", "sidebyside"]:
    candidates = glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/renders_{viz_type}_{SAVE_LABEL}", recursive=True)
    if not candidates:
        continue
    images = sorted(glob.glob(f"{candidates[0]}/*.png"))[:4]
    if not images:
        continue
    fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 5))
    if len(images) == 1:
        axes = [axes]
    for ax, p in zip(axes, images):
        ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8); ax.axis('off')
    plt.suptitle(f'{viz_type}: "{TEXT_QUERY}" (thr={THRESHOLD})', fontsize=14)
    plt.tight_layout(); plt.show()

if not any(glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/renders_*_{SAVE_LABEL}", recursive=True)):
    print("No activation images found.")

### 7-3. 다양한 쿼리 테스트

In [ ]:
# GT 카테고리 확인 (현재 씬)
import json
label_dir = os.path.join(DATASET_ROOT, "label", SCENE_NAME)
categories = set()
if os.path.exists(label_dir):
    for f in glob.glob(os.path.join(label_dir, "*.json")):
        with open(f) as fp:
            for obj in json.load(fp)["objects"]:
                categories.add(obj["category"])
    print(f"=== {SCENE_NAME} GT categories ({len(categories)}) ===")
    for c in sorted(categories):
        print(f"  - {c}")
else:
    print(f"[WARN] label dir not found: {label_dir}")

In [ ]:
# GT 카테고리 쿼리 (단어만)
QUERIES_GT = ["bowl", "chopsticks", "eggs", "egg", "kamaboko", "nori",
              "plate", "sake cup", "corn", "glass of water", "wavy noodles"]

# GT 카테고리 쿼리 (설명문)
QUERIES_GT_DESC = [
    "An object containing noodles and various toppings on a table with sloping edges",
    "A pair of utensils for holding food next to a yellow bowl",
    "Slices cut evenly in half down the center, above the ramen noodles",
    "Smooth slices with a delicate spiral pattern featuring a swirl pattern",
    "Long, dark green seaweed flakes placed against the side of the bowl",
    "A drinking utensils with a smooth surface near a sake bottle",
    "Slender strips of food in the center of the bowl, mixed with toppings",
]

# 자유 쿼리 (다양한 관점 테스트)
FREE_QUERIES = [
    "food",                    # 상위 카테고리
    "yellow object",           # 색상 기반
    "wooden object",           # 재질 기반
    "object on the left",      # 순수 공간 표현
    "bowl",                    # 짧은 단어 (GT와 중복이지만 의도적)
    "table",                   # 학습 대상이 아닌 배경 객체
    "car",                     # 씬에 없는 물체 (네거티브 테스트)
]

# 문장 쿼리
QUERIES_SENTENCE = [
    "a bowl of noodles",
    "something you eat with",
    "the hottest thing on the table",
    "a Japanese drink",
]

QUERIES = QUERIES_GT + QUERIES_GT_DESC + FREE_QUERIES + QUERIES_SENTENCE

for query in QUERIES:
    save_label = query.replace(" ", "_")[:40]
    print(f"\n>>> {query}")
    !cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python render_activation.py \
        -s {SCENE_PATH} -m {DRSPLAT_TRAINED_PATH} \
        --semantic_model sam --feature_level 1 --pq_index {PQ_INDEX} \
        --img_label "{query}" --img_save_label "{save_label}" \
        --threshold 0.5 -l language_features_dim3 2>&1 | tail -3
print("\nDone!")

In [ ]:
# 결과 시각화: side-by-side만 모아보기 (전체 쿼리가 많으므로)
n_queries = len(QUERIES)
fig, axes = plt.subplots(n_queries, 1, figsize=(20, 3*n_queries))
if n_queries == 1:
    axes = [axes]

for i, query in enumerate(QUERIES):
    save_label = query.replace(" ", "_")[:40]
    candidates = glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/renders_sidebyside_{save_label}", recursive=True)
    if candidates:
        images = sorted(glob.glob(f"{candidates[0]}/*.png"))
        if images:
            axes[i].imshow(Image.open(images[0]))
    label = query if len(query) <= 50 else query[:50] + "..."
    axes[i].set_title(label, fontsize=9, loc='left')
    axes[i].axis('off')

plt.suptitle(f"All Queries - {SCENE_NAME} (side-by-side: original | colormap | binary)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. 정량 평가 (mIoU)
GT 마스크(JSON 폴리곤)와 Dr. Splat activation 결과를 비교합니다.

In [ ]:
LABEL_DIR = os.path.join(DATASET_ROOT, "label")
EVAL_OUTPUT = os.path.join(WORK_DIR, "eval_results")

!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python evaluate_drsplat.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_TRAINED_PATH} \
    --pq_index {PQ_INDEX} \
    --label_dir {LABEL_DIR} \
    --output_dir {EVAL_OUTPUT} \
    --threshold 0.5 \
    -l language_features_dim3

In [ ]:
# 결과 로드
import json, cv2
import numpy as np
from collections import defaultdict

RESULTS_JSON = os.path.join(EVAL_OUTPUT, SCENE_NAME, "results.json")
with open(RESULTS_JSON) as f:
    ALL_RESULTS = json.load(f)

print(f"Total evaluations: {len(ALL_RESULTS)}")
print(f"mIoU: {np.mean([r['iou'] for r in ALL_RESULTS]):.2f}%")
print(f"Precision: {np.mean([r['precision'] for r in ALL_RESULTS]):.2f}%")
print(f"Recall: {np.mean([r['recall'] for r in ALL_RESULTS]):.2f}%")
print(f"F1: {np.mean([r['f1'] for r in ALL_RESULTS]):.2f}%")
print(f"Detection Rate: {np.mean([r['detected'] for r in ALL_RESULTS])*100:.1f}%")

# 카테고리별 결과
cat_results = defaultdict(list)
for r in ALL_RESULTS:
    cat_results[r["category"]].append(r["iou"])

print(f"\n{'Category':<25s} {'mIoU':>8s} {'Count':>6s}")
print("-" * 45)
for cat in sorted(cat_results.keys()):
    print(f"  {cat:<23s} {np.mean(cat_results[cat]):>7.2f}% {len(cat_results[cat]):>5d}")

In [ ]:
# 오버레이 시각화 (RGB + GT(초록) + Pred(빨강))
overlay_dir = os.path.join(EVAL_OUTPUT, SCENE_NAME, "overlay")
overlay_images = sorted(glob.glob(f"{overlay_dir}/*.png"))[:12]

if overlay_images:
    cols = min(4, len(overlay_images))
    rows = (len(overlay_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows == 1 or cols == 1:
        axes = list(axes)
    else:
        axes = list(axes.flat)
    for ax, p in zip(axes, overlay_images):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        fname = os.path.basename(p)
        ax.set_title(fname.rsplit('_iou', 1)[0].replace('_', ' '), fontsize=8)
        ax.axis('off')
    for ax in axes[len(overlay_images):]:
        ax.axis('off')
    plt.suptitle(f"Evaluation Overlay - {SCENE_NAME} (Green=GT, Red=Pred)", fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print("No overlay images found")

---
## 9. 문제점 분석 실험

### 9-1. 객체 크기별 성능 분석

In [ ]:
# 9-1: 객체 크기 vs IoU
areas = [r["area_ratio"] for r in ALL_RESULTS]
ious = [r["iou"] for r in ALL_RESULTS]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 산점도
ax1.scatter(areas, ious, alpha=0.6, edgecolors='k', linewidth=0.5)
ax1.set_xlabel("Object Size (% of image)"); ax1.set_ylabel("IoU (%)")
ax1.set_title("Object Size vs IoU")

# 크기 구간별 평균
bins = {"Tiny (<1%)": [], "Small (1-5%)": [], "Medium (5-15%)": [], "Large (>15%)": []}
for r in ALL_RESULTS:
    a = r["area_ratio"]
    if a < 1: bins["Tiny (<1%)"].append(r["iou"])
    elif a < 5: bins["Small (1-5%)"].append(r["iou"])
    elif a < 15: bins["Medium (5-15%)"].append(r["iou"])
    else: bins["Large (>15%)"].append(r["iou"])

names = list(bins.keys())
means = [np.mean(v) if v else 0 for v in bins.values()]
counts = [len(v) for v in bins.values()]
bars = ax2.bar(names, means, color=['#ff6b6b', '#ffa94d', '#69db7c', '#4dabf7'])
for bar, c in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'n={c}', ha='center', fontsize=9)
ax2.set_ylabel("Mean IoU (%)"); ax2.set_title("IoU by Object Size")

plt.suptitle("9-1: Object Size Analysis", fontsize=14); plt.tight_layout(); plt.show()

corr = np.corrcoef(areas, ious)[0, 1]
print(f"Pearson correlation (size vs IoU): {corr:.3f}")

### 9-2. 객체 유형별 성능 분석 (투명/반사/일반)

In [ ]:
# 9-2: 객체 유형별 분석
OBJECT_TYPE = {
    "glass of water": "transparent", "sake cup": "transparent",
    "chopsticks": "thin", "spoon": "thin",
    "wavy noodles": "thin", "nori": "thin",
}  # 나머지는 전부 "opaque"

type_results = defaultdict(list)
for r in ALL_RESULTS:
    otype = OBJECT_TYPE.get(r["category"], "opaque")
    type_results[otype].append(r["iou"])

fig, ax = plt.subplots(figsize=(8, 5))
types = sorted(type_results.keys())
means = [np.mean(type_results[t]) for t in types]
stds = [np.std(type_results[t]) for t in types]
colors = {"opaque": "#4dabf7", "thin": "#ffa94d", "transparent": "#ff6b6b"}
bars = ax.bar(types, means, yerr=stds, capsize=5, color=[colors.get(t, '#aaa') for t in types])
for bar, t in zip(bars, types):
    n = len(type_results[t])
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + stds[types.index(t)] + 1,
            f'n={n}', ha='center', fontsize=9)
ax.set_ylabel("Mean IoU (%)"); ax.set_title("9-2: IoU by Object Type")
plt.tight_layout(); plt.show()

for t in types:
    v = type_results[t]
    det = sum(1 for r in ALL_RESULTS if OBJECT_TYPE.get(r["category"], "opaque") == t and r["detected"])
    print(f"  {t:12s}: mIoU={np.mean(v):.1f}%, det={det}/{len(v)}")

### 9-3. 쿼리 유형별 성능 (단어 vs 설명문)

In [ ]:
# 9-3: 쿼리 길이/유형별 분석
# Dr. Splat은 카테고리 이름(단어)으로 쿼리하므로, 단어 길이와 IoU의 관계 분석
SPATIAL_KW = ["next to", "behind", "left", "right", "near", "between", "above", "below", "on top"]
ATTRIBUTE_KW = ["red", "green", "yellow", "round", "small", "large", "transparent", "dark", "smooth", "long"]

def classify_query(text):
    t = text.lower()
    has_sp = any(kw in t for kw in SPATIAL_KW)
    has_at = any(kw in t for kw in ATTRIBUTE_KW)
    if has_sp and has_at: return "mixed"
    elif has_sp: return "spatial"
    elif has_at: return "attribute"
    else: return "simple"

# sentence 필드가 있으면 분석, 없으면 category로 분석
query_types = defaultdict(list)
for r in ALL_RESULTS:
    text = r.get("sentence", r["category"])
    qtype = classify_query(text)
    query_types[qtype].append(r["iou"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 유형별 바 차트
qtypes = sorted(query_types.keys())
qmeans = [np.mean(query_types[q]) for q in qtypes]
qcounts = [len(query_types[q]) for q in qtypes]
bars = ax1.bar(qtypes, qmeans, color=['#69db7c', '#4dabf7', '#ffa94d', '#ff6b6b'])
for bar, c in zip(bars, qcounts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'n={c}', ha='center', fontsize=9)
ax1.set_ylabel("Mean IoU (%)"); ax1.set_title("IoU by Query Type")

# 쿼리 길이 vs IoU
lengths = [len(r["category"]) for r in ALL_RESULTS]
ax2.scatter(lengths, [r["iou"] for r in ALL_RESULTS], alpha=0.6, edgecolors='k', linewidth=0.5)
ax2.set_xlabel("Query Length (chars)"); ax2.set_ylabel("IoU (%)")
ax2.set_title("Query Length vs IoU")

plt.suptitle("9-3: Query Type Analysis", fontsize=14); plt.tight_layout(); plt.show()

### 9-4. 뷰 일관성 분석

In [ ]:
# 9-4: 동일 객체의 뷰별 IoU 일관성
obj_views = defaultdict(list)
for r in ALL_RESULTS:
    obj_views[r["category"]].append(r["iou"])

# 2개 이상 뷰에서 나타나는 객체만
multi_view = {k: v for k, v in obj_views.items() if len(v) >= 2}

fig, ax = plt.subplots(figsize=(12, 5))
cats = sorted(multi_view.keys())
for i, cat in enumerate(cats):
    vals = multi_view[cat]
    mean_v = np.mean(vals)
    ax.errorbar(i, mean_v, yerr=[[mean_v - min(vals)], [max(vals) - mean_v]],
                fmt='o', capsize=8, markersize=8, linewidth=2)
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=45, ha='right', fontsize=8)
ax.set_ylabel("IoU (%)"); ax.set_title("9-4: View Consistency (error bars = min/max across views)")
plt.tight_layout(); plt.show()

print(f"\n{'Category':<25s} {'Mean':>6s} {'Std':>6s} {'CV%':>6s} {'Views':>5s} {'Stability'}")
print("-" * 65)
for cat in cats:
    vals = multi_view[cat]
    m, s = np.mean(vals), np.std(vals)
    cv = (s / m * 100) if m > 0 else 999
    stab = "Stable" if cv < 30 else "Unstable" if cv < 80 else "Very Unstable"
    print(f"  {cat:<23s} {m:>5.1f}% {s:>5.1f}% {cv:>5.1f}% {len(vals):>4d}  {stab}")

### 9-5. 실패/성공 케이스 시각화

In [ ]:
# 9-5: 최악/최고 케이스 비교
sorted_results = sorted(ALL_RESULTS, key=lambda x: x["iou"])
worst = sorted_results[:5]
best = sorted_results[-5:][::-1]

fig, axes = plt.subplots(5, 4, figsize=(20, 25))
for i in range(5):
    # Worst: pred | gt
    if i < len(worst) and os.path.exists(worst[i].get("pred_path", "")):
        axes[i][0].imshow(Image.open(worst[i]["pred_path"]), cmap='gray')
        axes[i][0].set_title(f"WORST Pred: {worst[i]['category']}\nIoU={worst[i]['iou']:.1f}%", fontsize=8, color='red')
        axes[i][1].imshow(Image.open(worst[i]["gt_path"]), cmap='gray')
        axes[i][1].set_title(f"GT", fontsize=8)
    # Best: pred | gt
    if i < len(best) and os.path.exists(best[i].get("pred_path", "")):
        axes[i][2].imshow(Image.open(best[i]["pred_path"]), cmap='gray')
        axes[i][2].set_title(f"BEST Pred: {best[i]['category']}\nIoU={best[i]['iou']:.1f}%", fontsize=8, color='blue')
        axes[i][3].imshow(Image.open(best[i]["gt_path"]), cmap='gray')
        axes[i][3].set_title(f"GT", fontsize=8)
    for j in range(4):
        axes[i][j].axis('off')

plt.suptitle("9-5: Worst 5 (left) vs Best 5 (right)", fontsize=14)
plt.tight_layout(); plt.show()

# IoU 분포 히스토그램
fig, ax = plt.subplots(figsize=(8, 4))
all_ious = [r["iou"] for r in ALL_RESULTS]
ax.hist(all_ious, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(np.mean(all_ious), color='red', linestyle='--', label=f'Mean={np.mean(all_ious):.1f}%')
ax.axvline(np.median(all_ious), color='blue', linestyle='--', label=f'Median={np.median(all_ious):.1f}%')
ax.legend(); ax.set_xlabel("IoU (%)"); ax.set_ylabel("Count"); ax.set_title("IoU Distribution")
plt.tight_layout(); plt.show()

print(f"IoU=0 (complete fail): {sum(1 for x in all_ious if x == 0)}/{len(all_ious)}")
print(f"IoU>50 (good): {sum(1 for x in all_ious if x > 50)}/{len(all_ious)}")

### 9-6. 자유 텍스트 쿼리 vs GT 카테고리 쿼리 비교
학습에 사용되지 않은 임의의 텍스트(색상, 재질, 상위 카테고리 등)와 GT 카테고리의 IoU를 비교합니다.

In [ ]:
# 9-6: Dr. Splat은 CLIP 기반이므로 GT 카테고리 외의 쿼리도 가능
# 7-3에서 이미 시각화는 했으므로, 여기서는 GT가 있는 쿼리만 정량 비교
# (자유 쿼리는 GT가 없어서 IoU 계산 불가 → 시각적 분석만)

# GT 카테고리별 mIoU 요약 바 차트
cat_ious = defaultdict(list)
for r in ALL_RESULTS:
    cat_ious[r["category"]].append(r["iou"])

fig, ax = plt.subplots(figsize=(14, 5))
cats = sorted(cat_ious.keys(), key=lambda c: np.mean(cat_ious[c]), reverse=True)
means = [np.mean(cat_ious[c]) for c in cats]
colors_bar = ['#4dabf7' if m > 30 else '#ffa94d' if m > 10 else '#ff6b6b' for m in means]
bars = ax.bar(range(len(cats)), means, color=colors_bar)
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=45, ha='right', fontsize=8)
ax.set_ylabel("mIoU (%)")
ax.set_title("9-6: Per-Category mIoU (sorted by performance)")
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{m:.1f}', ha='center', fontsize=7)
plt.tight_layout(); plt.show()

### 9-7. 종합 분석 요약

In [ ]:
# 9-7: 종합 요약
all_ious = [r["iou"] for r in ALL_RESULTS]

print("=" * 70)
print(f"Dr. Splat 분석 종합 요약 - {SCENE_NAME}")
print("=" * 70)

print(f"""
[전체 성능]
  mIoU: {np.mean(all_ious):.2f}%  |  Median: {np.median(all_ious):.2f}%
  Detection Rate: {np.mean([r['detected'] for r in ALL_RESULTS])*100:.1f}%
  IoU=0 (실패): {sum(1 for x in all_ious if x==0)}/{len(all_ious)}
  IoU>50 (성공): {sum(1 for x in all_ious if x>50)}/{len(all_ious)}

[분석 1] 객체 크기 의존성
  - 큰 객체(>15%): 높은 IoU
  - 작은 객체(<1%): 낮은 IoU
  - 원인: CLIP feature의 granularity 한계 + SAM 마스크 품질

[분석 2] 객체 유형 한계
  - 투명/반사 객체: 3DGS 복원 한계
  - 얇은 객체: Gaussian 표현 한계

[분석 3] 쿼리 유형
  - Dr. Splat은 CLIP 기반이므로 단어/문장 모두 처리 가능
  - 공간 표현(spatial) 이해는 CLIP의 한계

[분석 4] 뷰 일관성
  - 같은 객체라도 뷰에 따라 IoU 편차 존재
  - 3D feature가 뷰 독립적이므로 이론적으로 일관적이어야 함
  - 편차가 크다면 PQ 압축 손실 또는 majority voting 한계

[분석 5] 성능 분포
  - IoU 분포가 양극화되는지 확인
  - "잘 되거나 안 되거나" → SAM 마스크 품질이 결정적

[Dr. Splat vs ReferSplat 비교 포인트]
  - Dr. Splat: 학습 없이 직접 등록 (빠름) + CLIP 기반 (open-vocab)
  - ReferSplat: gradient 최적화 (느림) + BERT 기반 (referring expression)
  - 같은 GT에서 mIoU 비교 가능
""")
print("=" * 70)

---
## 8. 체크포인트 자동 저장/복원 요약

| 시점 | 동작 |
|------|------|
| **3DGS 학습 전** | Drive에 체크포인트 있으면 자동 복원 |
| **3DGS 학습 후** | Drive에 자동 저장 |
| **Dr. Splat 후** | Drive에 자동 저장 |
| **런타임 재시작** | 셀 순서대로 실행하면 Drive에서 복원되어 학습 스킵 |

---
## Troubleshooting

| 문제 | 해결 |
|------|------|
| **OOM** | `TOPK`를 10~20으로 줄이기 |
| **submodules 없음** | `git submodule update --init --recursive` 실행 |
| **rasterizer 충돌** | 5단계에서 uninstall -> langsplat 설치 확인 |
| **PQ index 없음** | `ckpts/pq_index.faiss` 확인 |
| **런타임 재시작 후 데이터 없음** | Drive 백업에서 복원 또는 재다운로드 |